# AgroConecta - treinamento ACV no Colab
Selecione Ambiente de execução > Alterar tipo > GPU. Envie o ZIP do projeto. O notebook não contém resultados pré-inventados. Preserve uma cópia do modelo atual antes de treinar novamente.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile, pathlib
archive = next(pathlib.Path('.').glob('AgroConecta_ACV*.zip'))
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        dest = (pathlib.Path('/content') / item.filename).resolve()
        if not dest.is_relative_to('/content'):
            raise ValueError('Invalid archive path')
    z.extractall('/content')
%cd /content/agroconecta-acv

In [ ]:
%pip install -r requirements.txt

In [ ]:
!python scripts/prepare_data.py

In [ ]:
import torch
assert torch.cuda.is_available(), 'Ative a GPU do Colab ou use --device cpu.'
print(torch.cuda.get_device_name(0))
!python scripts/train.py --epochs 60 --imgsz 640 --batch 8 --device 0 --name colab_final

In [ ]:
!python scripts/evaluate.py --device 0

In [ ]:
import shutil
from pathlib import Path
Path('/content/export_acv').mkdir(exist_ok=True)
for folder in ['models','reports','runs']:
    shutil.copytree(folder, '/content/export_acv/'+folder, dirs_exist_ok=True)
shutil.make_archive('/content/AgroConecta_resultados', 'zip', '/content/export_acv')
files.download('/content/AgroConecta_resultados.zip')

Após baixar, copie models/best.pt e models/metadata.json para o projeto local e reinicie a API. Use o novo relatório de métricas; não mantenha conclusões do modelo anterior.